# LeetCode #1152 — Analyze User Website Visit Pattern
**Mức độ:** Medium (Premium) · **Chủ đề:** Sorting, Hash Map, `set`, Combinations, Counting · **Ngôn ngữ:** Python 3.

> **Tư duy cốt lõi:** Sắp xếp lịch sử theo thời gian → gom riêng mỗi người → liệt kê mọi chuỗi gồm 3 lần truy cập theo thứ tự → mỗi người chỉ đóng góp **1 điểm cho mỗi mẫu** → chọn mẫu nhiều người nhất, hòa thì lấy nhỏ nhất theo từ điển.

Notebook này dạy từ kiến thức Python cơ bản và cách duyệt các bộ ba tới thuật toán hoàn chỉnh và bài tự luyện.

## 1. Mục tiêu học tập

Sau bài này, bạn có thể:
1. Dùng `zip`, `sorted`, `dict`, `set`, `tuple`, `itertools.combinations`.
2. Phân biệt **thứ tự theo thời gian** với thứ tự phần tử ở ba mảng đầu vào.
3. Hiểu mẫu 3 website là **dãy con giữ thứ tự**, không bắt buộc liên tiếp; tên website được phép trùng nhau.
4. Phân biệt **đếm số người dùng khác nhau** với đếm số lần bộ ba xuất hiện.
5. Xử lý **hòa điểm theo thứ tự từ điển**, viết đúng chữ ký LeetCode và tự kiểm tra kết quả.

## 2. Đề bài

Có ba mảng cùng độ dài:
- `username[i]`: tên người dùng;
- `timestamp[i]`: thời điểm truy cập;
- `website[i]`: website được truy cập.

Một *pattern* là **ba tên website theo đúng thứ tự thời gian** mà một người từng truy cập (`a` rồi `b` rồi `c`). **Không bắt buộc** là ba lần truy cập liên tiếp và **không bắt buộc** ba website khác nhau.

**Score của pattern** = số lượng **người dùng phân biệt** đã đi qua pattern ấy ít nhất một lần. Trả pattern có score cao nhất; nếu hòa, trả pattern nhỏ nhất theo *lexicographical order* (thứ tự từ điển).

**Ràng buộc:** `3 <= N <= 50`; `timestamp[i]` thuộc `[1, 10^9]`; username/website gồm chữ cái thường và dài 1–10; có ít nhất một người đã truy cập từ ba lần trở lên. Mỗi bộ `(username, timestamp, website)` là duy nhất.

## 3. Ví dụ chính từ đề

```python
username = ["joe","joe","joe","james","james","james","james","mary","mary","mary"]
timestamp = [1,2,3,4,5,6,7,8,9,10]
website   = ["home","about","career","home","cart","maps","home","home","about","career"]
```

- `joe` đã đi qua `home → about → career`.
- `mary` cũng đã đi qua `home → about → career`.
- `james` có thể tạo nhiều pattern khác (ví dụ `home → cart → maps`), nhưng không có pattern nào của `james` trùng với pattern trên.

Kết quả: `['home','about','career']` vì score = **2 người**.

In [1]:
USERNAME = ['joe','joe','joe','james','james','james','james','mary','mary','mary']
TIMESTAMP = [1,2,3,4,5,6,7,8,9,10]
WEBSITE = ['home','about','career','home','cart','maps','home','home','about','career']
print('Có', len(USERNAME), 'dòng truy cập.')
for i in range(4):
    print(USERNAME[i], TIMESTAMP[i], WEBSITE[i])

Có 10 dòng truy cập.
joe 1 home
joe 2 about
joe 3 career
james 4 home


## 4. Python nền tảng: `zip` — ghép ba mảng

`zip(a, b, c)` ghép các phần tử **cùng chỉ số** thành từng *tuple* (bộ giá trị có thứ tự). Khi cần xử lý một lượt truy cập, ta thường muốn dữ liệu nằm cùng dòng, thay vì đọc ba mảng rời rạc.

Lưu ý: `zip` trả một iterator; dùng `list(...)` khi muốn xem hết. Tuple như `('joe',1,'home')` giữ cùng thông tin của ba mảng ở chỉ số 0.

In [2]:
users = ['lan','an','lan']
times = [9, 3, 5]
sites = ['cart','home','about']
rows = list(zip(users, times, sites))
print('Các bộ ba chưa sắp xếp:', rows)
assert rows[0] == ('lan', 9, 'cart')

Các bộ ba chưa sắp xếp: [('lan', 9, 'cart'), ('an', 3, 'home'), ('lan', 5, 'about')]


## 5. Python nền tảng: `sorted` — sắp theo thời gian

Trong đề, chỉ số `i` **không** được bảo đảm theo thứ tự thời gian. Hãy sắp `timestamp` trước.

Một mẹo là tạo tuple `(timestamp, username, website)` rồi dùng `sorted(...)`: Python ưu tiên phần tử đầu tiên (thời gian), sau đó đến các phần tử tiếp theo nếu cần. Với dữ liệu mẫu dưới, thứ tự thời gian là `3 → 5 → 9`.

In [3]:
events = sorted(zip(times, users, sites))
for time, user, site in events:
    print('t=',time,'user=',user,'site=',site)
assert [time for time,_,_ in events] == [3,5,9]

t= 3 user= an site= home
t= 5 user= lan site= about
t= 9 user= lan site= cart


## 6. Python nền tảng: `dict` và `defaultdict(list)`

Ta cần một danh sách website cho mỗi người: `history['lan'] = ['about','cart']`.

- `dict` lưu `key → value`; ở đây key là `username`, value là list website.
- `defaultdict(list)` tự tạo list rỗng khi gặp người dùng mới, giúp tránh kiểm tra `if user not in history`.
- `.append(site)` thêm site vào **cuối** list; vì ta duyệt sự kiện theo thời gian, list lưu đúng thứ tự.

In [4]:
from collections import defaultdict
history_demo = defaultdict(list)
for time, user, site in sorted(zip(times, users, sites)):
    history_demo[user].append(site)
print(dict(history_demo))
assert history_demo['lan'] == ['about','cart']

{'an': ['home'], 'lan': ['about', 'cart']}


## 7. Dãy con không nhất thiết liên tiếp

Với người dùng truy cập `['home','cart','about','career']`, các pattern đúng gồm:

- `('home','cart','about')`: dùng chỉ số `(0,1,2)`.
- `('home','about','career')`: dùng `(0,2,3)` — **bỏ qua `cart` vẫn hợp lệ**.
- `('cart','about','career')`: dùng `(1,2,3)`.

Ta chọn **3 chỉ số** `i < j < k`; đây là *combination* (tổ hợp) của ba vị trí, không phải hoán vị tùy ý. Với `m` lần truy cập, có `C(m,3) = m(m-1)(m-2)/6` bộ chỉ số như thế.

In [5]:
from itertools import combinations
sequence = ['home','cart','about','career']
for indices in combinations(range(len(sequence)), 3):
    pattern = tuple(sequence[i] for i in indices)
    print('index =', indices, '→', pattern)
assert len(list(combinations(range(4), 3))) == 4

index = (0, 1, 2) → ('home', 'cart', 'about')
index = (0, 1, 3) → ('home', 'cart', 'career')
index = (0, 2, 3) → ('home', 'about', 'career')
index = (1, 2, 3) → ('cart', 'about', 'career')


## 8. Vì sao dùng `tuple` làm pattern và `set` để loại trùng?

- `list` có thể thay đổi nên **không dùng làm key trong `dict` hoặc phần tử của `set`**.
- `tuple` như `('a','b','a')` là bất biến và có thể dùng làm key.
- `set` chứa mỗi giá trị **một lần**, nên ta có thể lấy tất cả pattern của **một user**, loại trùng, rồi tăng điểm cho từng pattern.

Ví dụ `['a','a','a','a']` có `C(4,3)=4` bộ chỉ số nhưng chỉ tạo **một** pattern tên website `('a','a','a')`; người dùng đó chỉ cho nó 1 điểm.

In [6]:
sites_one_user = ['a','a','a','a']
all_patterns = [tuple(x) for x in combinations(sites_one_user, 3)]
unique_patterns = set(all_patterns)
print('Theo chỉ số:', all_patterns)
print('Sau khi set:', unique_patterns)
assert len(all_patterns) == 4 and len(unique_patterns) == 1

Theo chỉ số: [('a', 'a', 'a'), ('a', 'a', 'a'), ('a', 'a', 'a'), ('a', 'a', 'a')]
Sau khi set: {('a', 'a', 'a')}


## 9. Cách dễ nghĩ (baseline): lưu tập người dùng cho mỗi pattern

Ta làm đầy đủ ba bước:
1. Sắp xếp mọi lượt truy cập theo thời gian và gom theo người.
2. Với mỗi người, sinh các bộ ba chỉ số `i<j<k`.
3. Lưu `users_by_pattern[pattern].add(user)` — vì là `set`, người giống nhau tự động chỉ được đếm một lần.

Cuối cùng chọn pattern có số người nhiều nhất. Cách này dễ kiểm tra trực giác nhưng giữ **tập username cho mỗi pattern**, tốn bộ nhớ hơn cách đếm trực tiếp bằng `Counter`.

In [7]:
def most_visited_by_user_sets(username, timestamp, website):
    history = defaultdict(list)
    for time, user, site in sorted(zip(timestamp, username, website)):
        history[user].append(site)
    users_by_pattern = defaultdict(set)
    for user, sites in history.items():
        for triple in combinations(sites, 3):
            users_by_pattern[triple].add(user)
    return list(min(users_by_pattern, key=lambda p: (-len(users_by_pattern[p]), p)))

print('Baseline:', most_visited_by_user_sets(USERNAME, TIMESTAMP, WEBSITE))
assert most_visited_by_user_sets(USERNAME, TIMESTAMP, WEBSITE) == ['home','about','career']

Baseline: ['home', 'about', 'career']


## 10. Điểm đột phá: mỗi user tự loại trùng, dùng `Counter`

Không cần ghi cả danh sách ai đã tạo pattern. Khi đang duyệt **từng người riêng biệt**, ta tạo:

```python
own_patterns = set(combinations(sites, 3))
```

Sau đó, với mỗi pattern trong set, tăng `score[pattern] += 1`. Set bảo đảm **mỗi người chỉ tăng điểm một lần** cho cùng pattern. `Counter` là `dict` chuyên đếm; key mới mặc định có giá trị 0.

Điểm quan trọng: nếu làm `for pattern in combinations(sites,3): score[pattern]+=1` mà **không có `set`**, bạn sẽ đếm **số bộ ba**, sai yêu cầu đếm người dùng phân biệt.

In [8]:
from collections import Counter
sites = ['a','b','a','b','a']
raw = list(combinations(sites,3))
unique = set(raw)
print('Số bộ ba chỉ số:',len(raw),'| Số pattern khác nhau:',len(unique))
print('Một pattern có thể xuất hiện nhiều lần:',raw.count(('a','b','a')))
assert raw.count(('a','b','a')) > 1

Số bộ ba chỉ số: 10 | Số pattern khác nhau: 7
Một pattern có thể xuất hiện nhiều lần: 4


## 11. So sánh trực tiếp đếm sai và đếm đúng

Với người `u` truy cập `a,b,a,b,a`, pattern `('a','b','a')` xuất hiện từ **nhiều** bộ chỉ số. Nhưng **score của pattern do u đóng góp chỉ bằng 1**.

In [9]:
wrong = Counter(combinations(['a','b','a','b','a'],3))
correct = Counter(set(combinations(['a','b','a','b','a'],3)))
print('Đếm số lần (SAI mục tiêu):', wrong[('a','b','a')])
print('Đếm 1 người (ĐÚNG):', correct[('a','b','a')])
assert wrong[('a','b','a')] > correct[('a','b','a')] == 1

Đếm số lần (SAI mục tiêu): 4
Đếm 1 người (ĐÚNG): 1


## 12. Thứ tự từ điển: Python so sánh các `tuple`

Python so sánh tuple theo thứ tự: phần tử đầu tiên khác nhau quyết định kết quả, giống cách tra từ điển.

Ví dụ `('a','b','a') < ('a','b','c')` vì hai vị trí đầu giống nhau, còn `'a' < 'c'`.

Khi chọn đáp án, ta cần ưu tiên **điểm cao nhất**, sau đó là **pattern nhỏ nhất**. Một công thức gọn:

```python
min(score, key=lambda p: (-score[p], p))
```

Vì `min` ưu tiên số nhỏ, **điểm lớn được đổi thành số âm nhỏ**. Khi bằng điểm, `p` quyết định thứ tự từ điển.

In [10]:
score_demo = Counter({('a','b','a'):2, ('a','b','c'):2, ('z','z','z'):1})
print('Thứ tự tuple:', ('a','b','a') < ('a','b','c'))
print('Chọn khi hòa điểm:', min(score_demo, key=lambda p: (-score_demo[p],p)))
assert min(score_demo, key=lambda p: (-score_demo[p],p)) == ('a','b','a')

Thứ tự tuple: True
Chọn khi hòa điểm: ('a', 'b', 'a')


## 13. Lời giải chuẩn dạng hàm

Bây giờ ghép các phần đã học:

- `sorted(zip(timestamp, username, website))` đưa về thứ tự thời gian;
- `defaultdict(list)` gom website theo mỗi người;
- `set(combinations(sites, 3))` tạo pattern duy nhất cho người đó;
- `Counter` ghi điểm;
- `min(..., key=...)` giải quyết cả score lẫn tie-break.

In [11]:
def most_visited_pattern(username, timestamp, website):
    history = defaultdict(list)
    for time, user, site in sorted(zip(timestamp, username, website)):
        history[user].append(site)

    score = Counter()
    for sites in history.values():
        patterns_of_this_user = set(combinations(sites, 3))
        for pattern in patterns_of_this_user:
            score[pattern] += 1

    winner = min(score, key=lambda p: (-score[p], p))
    return list(winner)

assert most_visited_pattern(USERNAME, TIMESTAMP, WEBSITE) == ['home','about','career']
print('Kết quả:', most_visited_pattern(USERNAME, TIMESTAMP, WEBSITE))

Kết quả: ['home', 'about', 'career']


## 14. Trace bằng tay qua từng người

Trên dữ liệu gốc:

| Người | Dãy website sau sắp xếp | Pattern đáng chú ý | Đóng góp |
|---|---|---|---|
| `joe` | `home → about → career` | `(home,about,career)` | +1 |
| `james` | `home → cart → maps → home` | `(home,cart,maps)`, `(home,cart,home)`, ... | Mỗi pattern +1 |
| `mary` | `home → about → career` | `(home,about,career)` | +1 |

Vậy `(home,about,career)` có điểm `2`; mọi pattern khác tối đa `1`. Chú ý `james` có 4 lượt truy cập nên tạo `C(4,3)=4` bộ chỉ số, **không phải** một bộ ba duy nhất.

In [12]:
def trace_patterns(username, timestamp, website):
    history = defaultdict(list)
    for time,user,site in sorted(zip(timestamp,username,website)):
        history[user].append(site)
    score = Counter()
    for user,sites in history.items():
        unique = set(combinations(sites,3))
        print(f'user={user}, sites={sites}, số pattern riêng={len(unique)}')
        for p in sorted(unique):
            score[p]+=1
            print('   ',p,'→ score tạm =',score[p])
    print('Đáp án:',list(min(score,key=lambda p:(-score[p],p))))

trace_patterns(USERNAME,TIMESTAMP,WEBSITE)

user=joe, sites=['home', 'about', 'career'], số pattern riêng=1
    ('home', 'about', 'career') → score tạm = 1
user=james, sites=['home', 'cart', 'maps', 'home'], số pattern riêng=4
    ('cart', 'maps', 'home') → score tạm = 1
    ('home', 'cart', 'home') → score tạm = 1
    ('home', 'cart', 'maps') → score tạm = 1
    ('home', 'maps', 'home') → score tạm = 1
user=mary, sites=['home', 'about', 'career'], số pattern riêng=1
    ('home', 'about', 'career') → score tạm = 2
Đáp án: ['home', 'about', 'career']


## 15. Code nộp LeetCode — đúng signature

Chữ ký của đề là `mostVisitedPattern(self, username: List[str], timestamp: List[int], website: List[str]) -> List[str]`.

Ta import mọi thành phần cần thiết ở cùng cell code, để có thể **copy riêng cell này** và nộp LeetCode ngay.

In [13]:
from typing import List
from collections import defaultdict, Counter
from itertools import combinations

class Solution:
    def mostVisitedPattern(self, username: List[str], timestamp: List[int], website: List[str]) -> List[str]:
        visits = defaultdict(list)
        for time, user, site in sorted(zip(timestamp, username, website)):
            visits[user].append(site)

        scores = Counter()
        for sites in visits.values():
            for pattern in set(combinations(sites, 3)):
                scores[pattern] += 1

        best = min(scores, key=lambda p: (-scores[p], p))
        return list(best)

sol = Solution()
assert sol.mostVisitedPattern(USERNAME,TIMESTAMP,WEBSITE) == ['home','about','career']
print('PASS ví dụ chính.')

PASS ví dụ chính.


## 16. Vì sao đúng? — ba lớp lập luận

1. **Không bỏ sót pattern hợp lệ:** với mọi người, `combinations(sites,3)` liệt kê đúng tất cả bộ ba chỉ số `i<j<k`, tức mọi dãy con 3 website theo thứ tự thời gian; không sinh thứ tự sai.
2. **Tính đúng score:** `set` gộp những lần xuất hiện trùng pattern **của một user**; `Counter` tăng 1 cho mỗi user có pattern đó. Vì mỗi user được duyệt một lượt, score sau cùng đúng bằng số người khác nhau.
3. **Chọn đúng đáp án:** `min` theo khóa `(-score,p)` tương đương tối đa hóa score, rồi tối thiểu hóa thứ tự từ điển khi hòa.

Do đó thuật toán luôn trả đáp án đúng trên các input đúng ràng buộc của đề.

## 17. Phân tích độ phức tạp

Gọi `N` là tổng số lượt truy cập (`N <= 50`), `m_u` là lượt truy cập của user `u`, và `P` là số pattern khác nhau được tạo ra.

- Sắp xếp tất cả lượt truy cập: `O(N log N)`.
- Với mỗi người, `combinations(sites,3)` tạo `C(m_u,3)` bộ ba. Tổng thời gian sinh và cập nhật: **`O(Σ_u C(m_u,3))` trung bình** đối với thao tác hash.
- Chọn pattern thắng: `O(P)` (giả sử độ dài tên website bị chặn theo đề).
- **Tổng:** `O(N log N + Σ_u C(m_u,3))`, trường hợp xấu `O(N³)` khi có một user với gần `N` lượt truy cập.
- **Bộ nhớ phụ:** `O(N + P + max_u C(m_u,3))` khi xét cả set tạm; có thể chặn trên bởi `O(N³)`.

Đề cho `N<=50`, nên việc liệt kê bộ ba là khả thi.

## 18. Kiểm thử ví dụ 2 — hòa điểm nhưng ưu tiên từ điển

Với `ua` đi qua `a,b,a` và `ub` đi qua `a,b,c`, mỗi pattern có score 1. Theo từ điển, `('a','b','a') < ('a','b','c')`, nên kết quả là `['a','b','a']`.

In [14]:
u2 = ['ua','ua','ua','ub','ub','ub']
t2 = [1,2,3,4,5,6]
w2 = ['a','b','a','a','b','c']
assert sol.mostVisitedPattern(u2,t2,w2) == ['a','b','a']
print('PASS ví dụ tie-break:',sol.mostVisitedPattern(u2,t2,w2))

PASS ví dụ tie-break: ['a', 'b', 'a']


## 19. Test phản ví dụ: thứ tự input đảo lộn, website trùng

Ba lỗi phổ biến:

1. Duyệt dữ liệu chưa sắp xếp → website bị đảo thứ tự.
2. Dùng `Counter(combinations(...))` không qua `set` → người có nhiều lượt trùng được cộng nhiều điểm.
3. Ép 3 website khác nhau → bỏ sót pattern hợp lệ như `('a','a','a')`.

Sau đây là test riêng cho từng lỗi.

In [15]:
# Đảo thứ tự các hàng đầu vào, không đổi thời điểm thật.
indices = list(range(len(USERNAME)-1,-1,-1))
assert sol.mostVisitedPattern([USERNAME[i] for i in indices],
                              [TIMESTAMP[i] for i in indices],
                              [WEBSITE[i] for i in indices]) == ['home','about','career']

# Một user tạo cùng pattern qua nhiều tổ hợp: score chỉ +1 cho user đó.
u = ['u','u','u','u','v','v','v']
t = [1,2,3,4,5,6,7]
w = ['a','b','c','c','a','b','c']
assert sol.mostVisitedPattern(u,t,w) == ['a','b','c']

# Cùng website được lặp ba lần vẫn là một pattern hợp lệ.
assert sol.mostVisitedPattern(['u']*3,[1,2,3],['a']*3) == ['a','a','a']
print('PASS thứ tự đảo, đếm user phân biệt, website lặp.')

PASS thứ tự đảo, đếm user phân biệt, website lặp.


## 20. Kiểm thử độc lập: liệt kê bộ ba trên dữ liệu gốc

Bộ kiểm thử sau **không gom trước theo user** như lời giải chính. Thay vào đó nó liệt kê các bộ ba lượt truy cập của toàn bộ lịch sử đã sắp xếp, chỉ giữ bộ ba có **cùng user**; mỗi pattern lưu các user vào một `set`.

Đây là cách rất chậm `O(N³)` nhưng **độc lập về cấu trúc** so với thuật toán sử dụng `combinations` theo user, giúp bắt lỗi khi viết code.

In [16]:
def oracle_by_global_triples(username,timestamp,website):
    logs = sorted(zip(timestamp,username,website))
    p_to_users = defaultdict(set)
    for i in range(len(logs)):
        for j in range(i+1,len(logs)):
            for k in range(j+1,len(logs)):
                ti,ui,wi = logs[i]
                tj,uj,wj = logs[j]
                tk,uk,wk = logs[k]
                if ui == uj == uk:
                    p_to_users[(wi,wj,wk)].add(ui)
    return list(min(p_to_users,key=lambda p:(-len(p_to_users[p]),p)))

assert oracle_by_global_triples(USERNAME,TIMESTAMP,WEBSITE) == ['home','about','career']
print('Oracle độc lập:',oracle_by_global_triples(USERNAME,TIMESTAMP,WEBSITE))

Oracle độc lập: ['home', 'about', 'career']


## 21. Đối chiếu ngẫu nhiên với oracle độc lập

Tạo lịch sử ngẫu nhiên nhỏ (cho phép website lặp) rồi **xáo trộn thứ tự các dòng**. Thời điểm dùng số nguyên khác nhau để bảo đảm thứ tự rõ ràng. Với mỗi input, so sánh lời giải nhanh và lời giải độc lập.

In [17]:
import random
rng = random.Random(1152)
checked = 0
for _ in range(180):
    u = []; t = []; w = []
    time = 1
    for user_idx in range(rng.randint(1,4)):
        count = rng.randint(3,5) if user_idx == 0 else rng.randint(0,5)
        for _ in range(count):
            u.append('user'+str(user_idx))
            t.append(time)
            w.append(rng.choice(['a','b','c']))
            time += rng.randint(1,4)
    perm = list(range(len(u)))
    rng.shuffle(perm)
    uu = [u[i] for i in perm]; tt = [t[i] for i in perm]; ww = [w[i] for i in perm]
    fast = sol.mostVisitedPattern(uu,tt,ww)
    slow = oracle_by_global_triples(uu,tt,ww)
    assert fast == slow, (uu,tt,ww,fast,slow)
    checked += 1
print('PASS',checked,'input ngẫu nhiên đối chiếu với oracle.')

PASS 180 input ngẫu nhiên đối chiếu với oracle.


## 22. Mẹo nhận dạng pattern và các biến thể

Hãy nhớ bốn dấu hiệu:
- **Dòng dữ liệu có timestamp** → sắp xếp trước khi phân tích hành vi theo thứ tự.
- **Nhóm theo người dùng** → `defaultdict(list)`.
- **Dãy con độ dài cố định** → `combinations(sequence, length)` (chọn vị trí tăng dần).
- **Đếm thực thể phân biệt** → tạo `set` trước khi tăng `Counter`.

Biến thể tự luyện: thay chuỗi 3 website thành chuỗi **2 website**; hoặc muốn tìm top-3 pattern, sắp `scores.items()` theo `(-score, pattern)`.

In [18]:
def all_distinct_2patterns(sites):
    return sorted(set(combinations(sites,2)))

assert all_distinct_2patterns(['a','b','a']) == [('a','a'),('a','b'),('b','a')]
print('Các 2-pattern:',all_distinct_2patterns(['a','b','a']))

Các 2-pattern: [('a', 'a'), ('a', 'b'), ('b', 'a')]


## 23. Tự luyện — 5 bài tăng dần

**Bài 1 (Python).** Ghép ba list thành các record rồi sắp xếp theo `timestamp`, trả list `(time,user,site)`.

**Bài 2 (tổ hợp).** Với các site `['a','b','a','c']`, liệt kê mọi pattern 3 website, **loại trùng** và sắp theo từ điển.

**Bài 3 (đếm điểm).** Viết hàm đếm pattern theo **user phân biệt**, đầu vào là dictionary `{username: [site1,site2,...]}` đã đúng thứ tự.

**Bài 4 (toàn bài).** Tự viết lại `mostVisitedPattern` mà không dùng code mẫu; có tie-break.

**Bài 5 (biến thể).** Trả về pattern **dài 2 website** nhiều người nhất (hòa thì chọn nhỏ nhất). Không cần 3 lượt truy cập, nhưng giả sử có ít nhất một người có 2 lượt.

In [19]:
# TODO: định nghĩa hàm hoàn chỉnh rồi tự bật test phía dưới.
def ex1_sort_events(username,timestamp,website):
    # TODO
    pass

def ex2_3patterns(sites):
    # TODO
    pass

def ex3_count_by_user(history):
    # TODO: return Counter có key là tuple 3 website
    pass

def ex4_most_visited(username,timestamp,website):
    # TODO
    pass

def ex5_best_2pattern(username,timestamp,website):
    # TODO
    pass

# Khi làm xong:
# assert ex2_3patterns(['a','b','a','c']) == [('a','a','c'),('a','b','a'),('a','b','c'),('b','a','c')]

## 24. Gợi ý theo 3 mức

**Mức 1 — hướng tư duy:** Bài nào cũng quay về 3 thao tác: **sắp theo thời gian**, **gom theo user**, **khử trùng pattern trong mỗi user**.

**Mức 2 — công cụ:** `sorted(zip(timestamp,username,website))`; `defaultdict(list)`; `set(combinations(sites,3))`; `Counter`.

**Mức 3 — gần code:** Với mỗi `sites`, chạy `for p in set(combinations(sites,3)): scores[p]+=1`. Đáp án toàn bài: `list(min(scores, key=lambda p:(-scores[p],p)))`. Bài 5 chỉ thay `3` thành `2`.

## 25. Đáp án tham khảo (sau khi đã tự làm)

Các hàm dưới đây cố tình giữ cách viết rõ ràng hơn là tối giản, để bạn có thể so sánh từng bước với TODO của mình.

In [20]:
def answer1_sort_events(username,timestamp,website):
    return sorted(zip(timestamp,username,website))

def answer2_3patterns(sites):
    return sorted(set(combinations(sites,3)))

def answer3_count_by_user(history):
    counts = Counter()
    for sites in history.values():
        for p in set(combinations(sites,3)):
            counts[p] += 1
    return counts

def answer4_most_visited(username,timestamp,website):
    return most_visited_pattern(username,timestamp,website)

def answer5_best_2pattern(username,timestamp,website):
    histories = defaultdict(list)
    for time,user,site in sorted(zip(timestamp,username,website)):
        histories[user].append(site)
    scores = Counter()
    for sites in histories.values():
        scores.update(set(combinations(sites,2)))
    return list(min(scores,key=lambda p:(-scores[p],p)))

assert answer1_sort_events(['u','v'],[3,1],['a','b']) == [(1,'v','b'),(3,'u','a')]
assert answer2_3patterns(['a','b','a','c']) == [('a','a','c'),('a','b','a'),('a','b','c'),('b','a','c')]
assert answer3_count_by_user({'u':['a','a','a','a'],'v':['a','a','a']})[('a','a','a')] == 2
assert answer4_most_visited(USERNAME,TIMESTAMP,WEBSITE) == ['home','about','career']
assert answer5_best_2pattern(['u']*2+['v']*2,[1,2,3,4],['a','b','a','b']) == ['a','b']
print('PASS cả 5 đáp án luyện tập.')

PASS cả 5 đáp án luyện tập.


## 26. Checklist submit và nhớ nhanh

**Mẫu tư duy:** `Sort → Group → Enumerate ordered triples → Deduplicate per user → Count distinct users → Tie-break`.

**Tự kiểm:**
- [ ] Có xử lý theo timestamp, không dựa thứ tự gốc.
- [ ] Cho phép website trùng lặp trong một pattern.
- [ ] Cho phép ba lần truy cập **không liên tiếp**.
- [ ] Mỗi user tối đa một điểm cho mỗi pattern.
- [ ] Hòa score chọn pattern **nhỏ nhất** từ điển.
- [ ] Trả `List[str]`, đúng signature `mostVisitedPattern`.

**Tài liệu đối chiếu:** [LeetCode #1152](https://leetcode.com/problems/analyze-user-website-visit-pattern/), [LeetCode Wiki #1152](https://leetcode.doocs.org/en/lc/1152/). Phần giải thích và bài tập bằng tiếng Việt được biên soạn theo cẩm nang đã cung cấp.